# Recurrent 1.5-Bit Concept Bottleneck Model: Coder Model Master Execution Notebook

This notebook runs the complete training and testing pipeline tailored specifically for a coding foundation model:
1. **Stage 1**: Supervised Fine-Tuning (PEFT/LoRA SFT) on `Qwen2.5-Coder-1.5B-Instruct` using Python code instruction dataset `iamtarun/python_code_instructions_18k_alpaca` via Unsloth.
2. **Stage 2**: Batched Layer Activation Hooking & Chunked Caching on Layer 14.
3. **Stage 3**: Coding-specific HybridCBM Representation Decomposition and Zero-Centered Pearson Concept Translation (configured with 30 static and 20 dynamic concepts).
4. **Stage 4**: T-TRM Recurrent Loop and CMR logic decider joint training under PST and monotonicity constraints.
5. **Stage 5**: Comprehensive Evaluation including neuro-symbolic tests and safe Dockerized HumanEval generative benchmarks.

## 1. Setup Environment & Repository

Verify workspace location and pull latest repository updates.

In [ ]:
import os

# Check if we are already in the repository root
if not os.getcwd().endswith("recurrent-1.5bit-cbm"):
    if not os.path.exists("recurrent-1.5bit-cbm"):
        print("Cloning repository...")
        !git clone https://github.com/Borisz42/recurrent-1.5bit-cbm.git
    
    # Navigate into the repository directory
    %cd recurrent-1.5bit-cbm
    print("Pulling latest repository updates...")
    !git pull
else:
    print("Already in repository root. Pulling latest updates...")
    !git pull

## 2. Install Dependencies

Install the optimized libraries. On Kaggle or Google Colab, Unsloth must be installed using their specific git repository.

In [ ]:
# Install optimized deep learning and training libraries
!pip install -q lightning pytorch-lightning transformers accelerate safetensors datasets trl bitsandbytes>=0.46.1
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Stage 1: Base Coder Model LoRA SFT

Fine-tune `Qwen2.5-Coder-1.5B-Instruct` in 4-bit precision on coding-specific instruct dataset `iamtarun/python_code_instructions_18k_alpaca`. We run for 1 epoch to adapt the model to clean Python code completions.

In [ ]:
# Run SFT on the Python code instruction dataset
!python src/system1/train_sft.py \
    --model_name "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" \
    --dataset "iamtarun/python_code_instructions_18k_alpaca" \
    --output_dir "./outputs_coder" \
    --adapter_dir "./adapters_coder" \
    --batch_size 8 \
    --gradient_accumulation_steps 2 \
    --epochs 1

## 4. Stage 2: Batched Coder Layer Activation Caching

Hook Layer 14 of the fine-tuned coder model and save intermediate activations. Activations are cached to `.safetensors` files of 100 samples each.

In [ ]:
# Run activation extraction over the coding dataset
!python src/system1/extract_activations.py \
    --model_name "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" \
    --adapter_dir "./adapters_coder" \
    --dataset "iamtarun/python_code_instructions_18k_alpaca" \
    --output_dir "./cached_coder_activations" \
    --layer_index 14 \
    --chunk_size 100 \
    --batch_size 16

## 5. Stage 3: Coding HybridCBM Optimization & Concept Translation

We load the cached coder activations and optimize the `HybridCBM` using 30 static coder concepts and 20 dynamic concepts to capture task-relevant representation details.

In [ ]:
import os
import torch
import torch.nn.functional as F
from safetensors.torch import load_file
from src.system1.hybrid_cbm import HybridCBM, CODER_CONCEPTS
from transformers import CLIPTokenizer, CLIPTextModel

device = "cuda" if torch.cuda.is_available() else "cpu"
cache_dir = "./cached_coder_activations"

# Load a representative subset of cached activations to prevent CPU RAM OOM
chunk_files = sorted([os.path.join(cache_dir, f) for f in os.listdir(cache_dir) if f.endswith(".safetensors")])[:50]
print(f"Loading {len(chunk_files)} chunk files...")

activations_list = []
for f in chunk_files:
    chunk_data = load_file(f)
    activations_list.append(chunk_data["activations"])
    
activations = torch.cat(activations_list, dim=0)
print(f"Loaded activations shape: {activations.shape} (stored on CPU)")

# Load CLIP text model to generate real static embeddings
print("Loading CLIP text model to generate real static embeddings...")
clip_model_name = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
text_model = CLIPTextModel.from_pretrained(clip_model_name).to(device)

def get_clip_embeddings(labels):
    inputs = tokenizer(labels, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = text_model(**inputs)
        embeddings = outputs.pooler_output
    return F.normalize(embeddings, p=2, dim=-1)

# Compute actual CLIP embeddings for coder concepts
print("Computing CLIP embeddings for coder concepts...")
static_embeddings = get_clip_embeddings(CODER_CONCEPTS)

# Initialize HybridCBM with 20 dynamic concepts and 30 coder concepts
n_dynamic = 20
hybrid_cbm = HybridCBM(
    n_dynamic=n_dynamic,
    clip_dim=512,
    emb_dim=activations.shape[-1],
    concepts=CODER_CONCEPTS,
    clip_embeddings=static_embeddings
).to(device)

# Setup Optimizer
optimizer = torch.optim.Adam(hybrid_cbm.parameters(), lr=1e-3)

# Mini-batch parameters
batch_size = 4096
n_samples = activations.shape[0]
model_dtype = hybrid_cbm.proj_clip.weight.dtype

print("Training HybridCBM representation decomposition...")
for epoch in range(100):
    permutation = torch.randperm(n_samples)
    epoch_loss = 0.0
    epoch_ortho_w = 0.0
    epoch_ortho_a = 0.0
    
    for i in range(0, n_samples, batch_size):
        optimizer.zero_grad()
        indices = permutation[i:i+batch_size]
        batch_x = activations[indices].to(device=device, dtype=model_dtype)
        
        z, x_rec, rec_loss = hybrid_cbm(batch_x)
        z_dynamic = z[:, hybrid_cbm.n_static:]
        
        # 1. Weight Orthogonality Regularization
        dynamic_weights = hybrid_cbm.decoder.weight[:, hybrid_cbm.n_static:].T
        dynamic_weights_norm = F.normalize(dynamic_weights, p=2, dim=-1)
        weight_similarity = torch.matmul(dynamic_weights_norm, dynamic_weights_norm.T)
        identity = torch.eye(n_dynamic, device=device)
        weight_ortho_loss = torch.sum((weight_similarity - identity) ** 2)
        
        # 2. Activation Orthogonality Regularization
        z_dynamic_centered = z_dynamic - z_dynamic.mean(dim=0, keepdim=True)
        z_dynamic_norm = F.normalize(z_dynamic_centered, p=2, dim=0)
        activation_similarity = torch.matmul(z_dynamic_norm.T, z_dynamic_norm)
        activation_ortho_loss = torch.sum((activation_similarity - identity) ** 2)
        
        total_loss = rec_loss + 0.1 * weight_ortho_loss + 0.5 * activation_ortho_loss
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += rec_loss.item() * len(indices)
        epoch_ortho_w += weight_ortho_loss.item() * len(indices)
        epoch_ortho_a += activation_ortho_loss.item() * len(indices)
        
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / n_samples
        avg_ortho_w = epoch_ortho_w / n_samples
        avg_ortho_a = epoch_ortho_a / n_samples
        with torch.no_grad():
            eval_x = activations[:1000].to(device=device, dtype=model_dtype)
            z_eval = torch.tanh(hybrid_cbm.proj_dynamic(hybrid_cbm.input_norm(eval_x)))
            stds = z_eval.std(dim=0).cpu().numpy()
            
        print(f"Epoch {epoch+1:03d} | Rec Loss: {avg_loss:.6f} | Weight Ortho: {avg_ortho_w:.6f} | Activation Ortho: {avg_ortho_a:.6f} | Stds: [{', '.join([f'{s:.4f}' for s in stds])}]")

# Save HybridCBM checkpoint
torch.save(hybrid_cbm.state_dict(), "./hybrid_cbm_coder.pt")
print("Saved Coder HybridCBM checkpoint to ./hybrid_cbm_coder.pt")

### Concept Translation

We project the learned dynamic concepts into the Candidate Concept Bank via CLIP space cosine similarity to assign human-understandable labels.

In [ ]:
# Zero-Centered Pearson Correlation Concept Translation
candidate_labels = [
    "syntax validation", "variable scoping", "control flow", "data structures",
    "algorithmic efficiency", "exception handling", "modular function design",
    "string formatting", "mathematical computation", "file and resource io",
    "regular expression", "recursive algorithm", "sorting and searching",
    "memory allocation", "object oriented", "type verification",
    "input validation", "database querying", "concurrency thread",
    "pointer reference", "network communication", "boolean logic",
    "bitwise operation", "compiler parsing", "version control",
    "unit testing", "debugging logic", "api endpoint",
    "data serialization", "inheritance class", "hash table",
    "linked list", "binary tree", "memory leak", "stack overflow"
]

print("Computing CLIP embeddings for candidate labels...")
candidate_embeddings = get_clip_embeddings(candidate_labels)

with torch.no_grad():
    model_dtype = hybrid_cbm.proj_clip.weight.dtype
    eval_size = min(10000, activations.shape[0])
    activations_sub = activations[:eval_size].to(device=device, dtype=model_dtype)
    
    x_clip = hybrid_cbm.proj_clip(hybrid_cbm.input_norm(activations_sub))
    x_clip_norm = F.normalize(x_clip, p=2, dim=-1)
    
    z_dynamic = torch.tanh(hybrid_cbm.proj_dynamic(hybrid_cbm.input_norm(activations_sub)))
    
    z_dynamic_centered = z_dynamic - z_dynamic.mean(dim=0, keepdim=True)
    z_dynamic_norm = F.normalize(z_dynamic_centered, p=2, dim=0)
    
    candidate_embeds_norm = F.normalize(candidate_embeddings.to(dtype=model_dtype), p=2, dim=-1)
    candidate_activations = torch.matmul(x_clip_norm, candidate_embeds_norm.T)
    
    candidate_activations_centered = candidate_activations - candidate_activations.mean(dim=0, keepdim=True)
    candidate_activations_norm = F.normalize(candidate_activations_centered, p=2, dim=0)
    
    correlation_matrix = torch.matmul(z_dynamic_norm.T, candidate_activations_norm)
    abs_correlation = torch.abs(correlation_matrix)
    max_abs_corrs, max_indices = torch.max(abs_correlation, dim=-1)

print("Translated Coder Dynamic Concepts:")
threshold = 0.35
for i in range(hybrid_cbm.n_dynamic):
    idx = max_indices[i].item()
    val = correlation_matrix[i, idx].item()
    if abs(val) < threshold:
        label = "unmapped"
    elif val < 0:
        label = f"not {candidate_labels[idx]}"
    else:
        label = candidate_labels[idx]
    print(f"  Dynamic Concept {i+1} -> Label: '{label}' (Correlation: {val:.4f})")

## 6. Stage 4: T-TRM Loop & Rule-Memory Coder Joint Training

We jointly optimize the `TTRMLoop` (PST logic gate parameters) and the `CMRModel` decider on coder activations using the customized `--concepts_type coder` and `--n_dynamic 20` arguments.

In [ ]:
# Jointly train the T-TRM loop and CMR decider on the cached coder activations
!python -u src/t_trm/train_trm.py \
    --cache_dir "./cached_coder_activations" \
    --hybrid_cbm_path "./hybrid_cbm_coder.pt" \
    --concepts_type coder \
    --n_dynamic 20 \
    --output_dir "./t_trm_coder_outputs" \
    --epochs 50 \
    --batch_size 1024 \
    --max_chunks 20

## 7. Stage 5: Evaluation & Safe Coding Benchmarks

Now we run the dual evaluation suite: the neuro-symbolic classification testing suite, followed by the safe Dockerized HumanEval generative coding benchmark.

In [ ]:
# Run the formal neuro-symbolic testing suite under the coding concepts context
!python -u src/eval/run_tests.py \
    --cache_dir "./cached_coder_activations" \
    --hybrid_cbm_path "./hybrid_cbm_coder.pt" \
    --model_dir "./t_trm_coder_outputs" \
    --concepts_type coder \
    --n_dynamic 20 \
    --max_chunks 20

### Safe Dockerized HumanEval Benchmark

Compare the base coder model performance vs the SFT fine-tuned model inside a secure Docker container environment.

In [ ]:
print("Evaluating Base Coder Model on HumanEval...")
!python -u src/eval/run_coder_eval.py \
    --model_name "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" \
    --output_dir "./eval_outputs_base"

print("\nEvaluating SFT Fine-Tuned Coder Model on HumanEval...")
!python -u src/eval/run_coder_eval.py \
    --model_name "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" \
    --adapter_dir "./adapters_coder" \
    --output_dir "./eval_outputs_sft"

### Kaggle / Non-Docker Direct Evaluation Fallback

If you are running this notebook in an environment without Docker (such as Kaggle or Google Colab), use the `--no_docker` flag to generate completions, and then run the evaluator directly using the python `human-eval` package.

In [ ]:
# 1. Generate completions without Docker
print("Generating completions for Base Model...")
!python -u src/eval/run_coder_eval.py \
    --model_name "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" \
    --no_docker \
    --output_dir "./eval_outputs_base"

print("\nGenerating completions for SFT Model...")
!python -u src/eval/run_coder_eval.py \
    --model_name "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit" \
    --adapter_dir "./adapters_coder" \
    --no_docker \
    --output_dir "./eval_outputs_sft"

# 2. Install official OpenAI human-eval package
!pip install -q git+https://github.com/openai/human-eval.git

# 3. Run evaluation directly in the current sandbox environment
import os
os.environ["ALLOW_CODE_EXECUTION"] = "1"

print("\n--- Evaluating Base Model ---")
!python -m human_eval.evaluate_functional_correctness ./eval_outputs_base/humaneval_completions.jsonl

print("\n--- Evaluating SFT Model ---")
!python -m human_eval.evaluate_functional_correctness ./eval_outputs_sft/humaneval_completions.jsonl

In [ ]:
import json
import os

def print_humaneval_summary(results_file):
    if not os.path.exists(results_file):
        print(f"Results file {results_file} not found.")
        return
    passed = 0
    failed = 0
    errors = {}
    with open(results_file, "r", encoding="utf-8") as rf:
        for line in rf:
            try:
                data = json.loads(line.strip())
                res = data.get("result", "")
                if res == "passed":
                    passed += 1
                else:
                    failed += 1
                    if "AssertionError" in res:
                        err_type = "AssertionError"
                    elif "SyntaxError" in res:
                        err_type = "SyntaxError"
                    elif "NameError" in res:
                        err_type = "NameError"
                    elif "TypeError" in res:
                        err_type = "TypeError"
                    elif "IndexError" in res:
                        err_type = "IndexError"
                    elif "KeyError" in res:
                        err_type = "KeyError"
                    elif "Timeout" in res or "timed-out" in res.lower() or "time out" in res.lower():
                        err_type = "Timeout"
                    else:
                        err_type = res.split("\n")[0][:40] if "\n" in res else res[:40]
                    errors[err_type] = errors.get(err_type, 0) + 1
            except Exception:
                pass
    total_eval = passed + failed
    if total_eval > 0:
        print(f"\nResults for {os.path.basename(results_file)}:")
        print(f"Total Evaluated Tasks: {total_eval}")
        print(f"Passed:                {passed} ({passed/total_eval*100:.2f}%)")
        print(f"Failed:                {failed} ({failed/total_eval*100:.2f}%)")
        if failed > 0:
            print("Failure Breakdown:")
            for err_type, count in sorted(errors.items(), key=lambda x: x[1], reverse=True):
                print(f"  - {err_type}: {count} ({count/failed*100:.1f}%)")
    print("=" * 50)

print_humaneval_summary("./eval_outputs_base/humaneval_completions.jsonl_results.jsonl")
print_humaneval_summary("./eval_outputs_sft/humaneval_completions.jsonl_results.jsonl")

## 8. Steered Inference & Verification on Custom Queries

We test the end-to-end neuro-symbolic pipeline on three custom coding queries. For each query, we output the System 1 causal generation alongside the System 2 Concept Activations and Decider Rules.

In [ ]:
import argparse
from src.eval.chatbot_app import SteeredChatbot

# Configure arguments to match Stage 4/5 configuration
args = argparse.Namespace(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    adapter_dir="./adapters_coder",
    hybrid_cbm_path="./hybrid_cbm_coder.pt",
    model_dir="./t_trm_coder_outputs",
    n_dynamic=20,
    clip_dim=512,
    n_latent=8,
    n_rules=10,
    layer_index=14,
    max_tokens=256,
    load_in_4bit=True,
    concepts_type="coder",
    cli=True
)

# Instantiate the chatbot engine
print("Initializing SteeredChatbot...")
engine = SteeredChatbot(args)

# Define 3 specific coding queries
coder_queries = [
    "Write a python function to compute the Fibonacci sequence using recursion.",
    "Write a function that validates if a given string is a valid email address using regex.",
    "Implement a class representing a binary search tree with an insert method."
]

# Run and log details for each query
for idx, query in enumerate(coder_queries):
    print(f"\n=======================================================")
    print(f"QUERY {idx+1}: {query}")
    print(f"=======================================================")
    
    response, concepts, rules = engine.generate_and_reason(query)
    
    print("\n[System 1 Output (Generated Response)]")
    print(response.strip())
    
    print("\n[System 2 Debugger - Concept Activations]")
    for name, val in sorted(concepts.items(), key=lambda x: x[1], reverse=True)[:10]:
        bar = "█" * int(val * 10) + "░" * (10 - int(val * 10))
        print(f"  - {name:30s}: {bar} ({val*100:5.1f}%)")
        
    print("\n[System 2 Debugger - Active Logic Constraints]")
    print(rules)
    print("-------------------------------------------------------")